# 🎯 Job Acceptance Prediction System
## Phase 2: Initial Data Ingestion & Comprehensive Audit

**Objective:** Profile the raw recruitment dataset (51,500 records, 26 features), audit data types, analyze missing value distributions, inspect duplicate records, detect categorical inconsistencies, and evaluate target class balance (`status`).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options for comprehensive inspection
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda x: "%.2f" % x)

DATA_PATH = "../dataset/HR_Job_Placement_Dataset.csv"
print(f"Dataset exists: {os.path.exists(DATA_PATH)}")

### 1. Load Dataset & Dimension Verification

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Total Rows: {df.shape[0]:,}")
print(f"Total Features: {df.shape[1]}")
df.head(5)

### 2. Schema, Data Types & Memory Profiling

In [ ]:
df.info()

### 3. Missing Value Distribution Audit

In [ ]:
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df)) * 100
missing_summary = pd.DataFrame({
    "Missing_Count": null_counts,
    "Percentage_%": null_pct
}).sort_values(by="Missing_Count", ascending=False)

print("--- Columns with Missing Values ---")
missing_summary[missing_summary["Missing_Count"] > 0]

### 4. Duplicate Records Audit

In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate Rows Found: {duplicate_count:,} ({duplicate_count / len(df) * 100:.2f}%)")

### 5. Categorical Feature Consistency & Whitespace Inspection

In [ ]:
cat_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
print(f"Categorical Columns ({len(cat_cols)}): {cat_cols}\n")

for col in cat_cols:
    uniques = df[col].dropna().unique().tolist()
    print(f"Feature: '{col}' (Total unique: {len(uniques)})")
    print(f"  Values: {uniques}\n")

### 6. Target Variable Analysis (`status`)

In [ ]:
status_dist = df["status"].value_counts(dropna=False)
status_pct = df["status"].value_counts(normalize=True) * 100
target_df = pd.DataFrame({"Count": status_dist, "Percentage_%": status_pct})
print(target_df)

### 7. Numerical Summary & Outlier Inspection

In [ ]:
df.describe().T

### 8. Key Audit Findings & Strategy for Phase 3

1. **Dimensions:** Raw dataset contains **51,500 rows** and **26 columns**.
2. **Duplicates:** **1,376 duplicate rows (2.67%)** detected that must be dropped.
3. **Missing Values:**
   - High nulls (~16.4%): `relocation_willingness`, `layoff_history`, `career_switch_willingness`, `relevant_experience`
   - Moderate nulls (~11.9%): `job_role_match`
   - Low/Numeric nulls (2-5%): `ssc_percentage`, `hsc_percentage`, `notice_period_days`, `employment_gap_months`
4. **Categorical Inconsistencies:**
   - `gender`: Contains mixed casing (`Male`, `Female`, `female`, `male`).
   - `internship_experience`: Contains irregular casing and trailing spaces (`No`, `yes `, `Yes`, ` no`).
   - `company_tier`: Trailing spaces present (`Tier 1 `, `Tier 2 `, `Tier 3 `).
5. **Target Imbalance:** ~69.8% `Not Placed` vs ~30.2% `Placed`. Stratified sampling is essential for modeling.